### Imports

In [1]:
import os, warnings, re, json, time, hashlib
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd

from llama_index.core import (
    SimpleDirectoryReader, VectorStoreIndex, StorageContext,
    Settings, PromptTemplate, load_index_from_storage,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import QueryBundle
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
with warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)   
    from ragas.metrics import (
        faithfulness, answer_relevancy, context_precision,
        context_recall, answer_similarity,
    )
from langchain_ollama import ChatOllama, OllamaEmbeddings
import re

c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Config

In [2]:
CONFIG = {
    "run_name": "naive_rag_v1",
    "generator": "llama3.2:3b",
    "embed_model": "nomic-embed-text",
    "chunk_size": 512,
    "chunk_overlap": 50,
    "top_k": 5,
    # retrieval depth for recall@k              
    "retrieval_k_max": 10,    
    "prompt_version": "v1",
    "judge_model": "llama3.1:8b",
}

DATA_DIR     = Path("./data_sample")
TESTSET_PATH = Path("./testset.jsonl")
RUNS_DIR     = Path("./runs")
FAISS_ROOT   = Path("./faiss_index")

# Testing period for now
EVAL_SPLIT    = "tuning"   
ALLOW_HELDOUT = False      

RUN_RAGAS           = True
RAGAS_MAX_QUESTIONS = 40
USE_MLFLOW          = False

ANSWER_OK_THRESHOLD = 0.7
# recall@k depths
KS = (1, 3, 5, 10)
# >= 50% of a gold span      
RELEVANT_FRAC    = 0.5
# top-k chunks altogether cover >= 80% of it      
SPAN_COVERED_FRAC = 0.8

RUNS_DIR.mkdir(exist_ok=True)
assert EVAL_SPLIT == "tuning" or ALLOW_HELDOUT
# test hyper-parameter hash tracker
def sha_short(*parts):
    return hashlib.sha256("|".join(map(str, parts)).encode()).hexdigest()[:10]
# Dataset hash tracker
def corpus_hash(data_dir):
    h = hashlib.sha256()
    for p in sorted(Path(data_dir).glob("*.txt")):
        h.update(p.name.encode()); h.update(p.read_bytes())
    return h.hexdigest()[:10]

CORPUS_HASH  = corpus_hash(DATA_DIR)
TESTSET_HASH = hashlib.sha256(TESTSET_PATH.read_bytes()).hexdigest()[:10]
print("corpus", CORPUS_HASH, "| testset", TESTSET_HASH)


corpus b1830d2859 | testset fb0e56598b


### Fact filter

In [ ]:
# Filter contradictory facts
def audit_terms(patterns, data_dir=DATA_DIR):
    for p in sorted(Path(data_dir).glob("*.txt")):
        for n, line in enumerate(p.read_text(encoding="utf-8").splitlines(), 1):
            if all(re.search(pt, line, re.I) for pt in patterns):
                print(f"{p.name}:{n}: {line.strip()}")

audit_terms(["cycle"])
print()
audit_terms([r"\bDOD\b|depth of discharge", r"\d"])

battery_storage.txt:16: - Cycle life: 2,000-3,000 cycles at 80% depth of discharge
battery_storage.txt:25: - Cycle life: 1,000-2,000 cycles at 80% depth of discharge
battery_storage.txt:33: - Cycle life: 300-500 cycles at 50% depth of discharge
battery_storage.txt:60: (DOD) and cycle life follows approximately:
battery_storage.txt:62: - 20% DOD: 8,000+ cycles
battery_storage.txt:63: - 40% DOD: 5,000+ cycles
battery_storage.txt:64: - 60% DOD: 3,000+ cycles
battery_storage.txt:65: - 80% DOD: 2,000-3,000 cycles
battery_storage.txt:66: - 100% DOD: 800-1,200 cycles (not recommended)
battery_storage.txt:93: - Expected lifetime at 50% daily DOD: approximately 4,000 cycles,

battery_storage.txt:16: - Cycle life: 2,000-3,000 cycles at 80% depth of discharge
battery_storage.txt:25: - Cycle life: 1,000-2,000 cycles at 80% depth of discharge
battery_storage.txt:33: - Cycle life: 300-500 cycles at 50% depth of discharge
battery_storage.txt:57: 4. DEPTH OF DISCHARGE AND LIFETIME
battery_storage.txt:

### RAG Pipeline

In [2]:
model = Ollama(model = "llama3.2:3b", temperature = 0.0, request_timeout = 180.0)
embedding = OllamaEmbedding(model_name="nomic-embed-text")
Settings.llm = model
Settings.embed_model = embedding

# Setting up non-volatile Faiss DB
faiss_dir = "./faiss_index"
os.makedirs(faiss_dir, exist_ok=True)

# Reloading the index instead of re-embedding
if os.path.exists(os.path.join(faiss_dir, "docstore.json")):
    vector_store = FaissVectorStore.from_persist_dir(faiss_dir)
    storage_context = StorageContext.from_defaults(vector_store=vector_store, persist_dir=faiss_dir)
    index = load_index_from_storage(storage_context)
else:
    documents = SimpleDirectoryReader(input_dir="./data_sample").load_data()
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=50).get_nodes_from_documents(documents)

    dim = len(embedding.get_text_embedding("test"))          
    vector_store = FaissVectorStore(faiss_index=faiss.IndexFlatL2(dim))   
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    index.storage_context.persist(persist_dir=faiss_dir)

qa_prompt = PromptTemplate(
    "Context information:\n{context_str}\n\n"
    "Answer the question using ONLY the context above. Be complete, and copy numbers, "
    "ranges and units exactly as written. If several chemistries/items are in the context, "
    "compare them. If the answer is not in the context, say \"I don't know\".\n"
    "Question: {query_str}\nAnswer: "
)      

query_engine = index.as_query_engine(similarity_top_k=5, text_qa_template=qa_prompt)

### Result

In [4]:
results = []

for q, gt in zip(questions, ground_truths):
    r = query_engine.query(q)
    results.append({
        "question": q,
        "answer": str(r.response).strip(),
        "ground_truth": gt,
        "contexts": [n.get_content() for n in r.source_nodes],   
    })
    print("=" * 60, "\nQ:", q, "\nA:", results[-1]["answer"])

Q: What is the optimal tilt angle for a site at 40 degrees north latitude? 
A: The optimal tilt angle for a site at 40 degrees north latitude is 40 degrees from horizontal.
Q: How often should solar panels be cleaned and what should be used? 
A: According to the context, solar panels should be cleaned quarterly. The text states:

"Quarterly: Clean panels with a soft brush and deionised water. Never use abrasive cleaners or pressure washers above 40 bar, as these can crack the tempered glass cover."
Q: What is the maximum pressure washer pressure allowed when cleaning solar panels? 
A: 40 bar.
Q: How does shading affect solar panel output and how should it be assessed? 
A: Shading can reduce the output of an entire string of solar panels by up to 80%. A shading study using a solace sensor or PV*SOL software must be performed at three times of day (9 AM, noon, 3 PM) across equinox and solstice dates.
Q: What roof load capacity is required for residential solar installations? 
A: At least

#### Context-coverage score

In [5]:
# Measuring word overlap percentage between document passage and answer
def numbers(t):
    return {n.replace(",", "").rstrip(".") for n in re.findall(r"\d+(?:[.,]\d+)*", t)}

def fact_score(answer, truth):
    t = numbers(truth)
    return len(t & numbers(answer)) / len(t) if t else float("nan")

tok = lambda t: set(re.findall(r"[a-z0-9\.\-]+", t.lower()))
for r, g in zip(results, gold_contexts):
    gold, got = tok(g), tok(" ".join(r["contexts"]))
    print(f"{len(gold & got)/len(gold):.0%}  {r['question']}")

100%  What is the optimal tilt angle for a site at 40 degrees north latitude?
100%  How often should solar panels be cleaned and what should be used?
100%  What is the maximum pressure washer pressure allowed when cleaning solar panels?
100%  How does shading affect solar panel output and how should it be assessed?
100%  What roof load capacity is required for residential solar installations?
100%  How much energy should a 6 kW system produce per day at 4.2 peak sun hours in a moderate climate?
100%  How much does panel output drop with temperature?
97%  What is the minimum DC cable cross-section for runs under 20 metres, and what are the inverter output cable requirements?
100%  What does fault code F-003 mean and what should be done?
100%  What temperature range can lithium iron phosphate batteries operate in?
100%  Which battery type has the lowest initial cost but the highest lifetime cost, and why?
92%  How does the energy density of NMC batteries compare to other chemistries?
98%

### Evaluation

In [6]:

JUDGE_MODEL = "llama3.1:8b"

judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL, temperature=0, num_ctx=8192))
judge_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))


answer_relevancy.strictness = 1

eval_dataset = EvaluationDataset.from_list([
    {
        "user_input": r["question"],
        "response": r["answer"],
        "retrieved_contexts": r["contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

scores = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall, answer_similarity],
    llm=judge_llm,
    embeddings=judge_emb,
    run_config=RunConfig(timeout=600, max_workers=1, max_retries=3),   # local models: go sequential
)

df = scores.to_pandas()
df.to_csv("ragas_scores.csv", index=False)
print(df.select_dtypes("number").mean().round(3))

pd.set_option("display.max_colwidth", 60)
cols = ["user_input", "faithfulness", "answer_relevancy", "context_precision", "context_recall", "semantic_similarity"]
df[[c for c in cols if c in df.columns]].round(2)

C:\Users\User\AppData\Local\Temp\ipykernel_22676\4014697071.py:3: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatOllama(model=JUDGE_MODEL, temperature=0, num_ctx=8192))
C:\Users\User\AppData\Local\Temp\ipykernel_22676\4014697071.py:4: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="nomic-embed-text"))
Evaluating: 100%|██████████| 85/85 [10:59<00:00,  7.76s/it]


faithfulness         0.862
answer_relevancy     0.818
context_precision    0.952
context_recall       0.961
answer_similarity    0.859
dtype: float64


,user_input,faithfulness,answer_relevancy,context_precision,context_recall
0,What is the optimal tilt angle for a site at 40 degrees ...,1.00,1.00,1.00,1.00
1,How often should solar panels be cleaned and what should...,1.00,0.97,1.00,1.00
2,What is the maximum pressure washer pressure allowed whe...,1.00,0.55,0.70,1.00
3,How does shading affect solar panel output and how shoul...,0.12,0.68,1.00,1.00
4,What roof load capacity is required for residential sola...,1.00,0.73,1.00,1.00
5,How much energy should a 6 kW system produce per day at ...,1.00,0.97,0.89,0.33
6,How much does panel output drop with temperature?,0.50,0.64,1.00,1.00
7,What is the minimum DC cable cross-section for runs unde...,1.00,0.92,1.00,1.00
8,What does fault code F-003 mean and what should be done?,0.67,0.81,1.00,1.00
9,What temperature range can lithium iron phosphate batter...,1.00,0.95,1.00,1.00


#### RAG vs baseline-LLM

In [7]:
baseline_answers = [str(model.complete(f"Answer concisely: {q}")).strip() for q in questions]
rag_answers = [r["answer"] for r in results]

for q, b, a in zip(questions[:9], baseline_answers[:9], rag_answers[:9]):
    print("Q:", q, "\n  baseline:", b, "\n  RAG     :", a, "\n")

Q: What is the optimal tilt angle for a site at 40 degrees north latitude? 
  baseline: The optimal tilt angle for a solar panel system at 40 degrees north latitude is typically between 30° and 40°. This allows for optimal energy production while considering snow load and shading. 
  RAG     : The optimal tilt angle for a site at 40 degrees north latitude is 40 degrees from horizontal. 

Q: How often should solar panels be cleaned and what should be used? 
  baseline: Solar panels should be cleaned every 6-12 months, depending on the location and environmental conditions. A soft, dry cloth or a microfiber cloth is recommended for cleaning, as it won't scratch the panels. Avoid using:

* Harsh chemicals or cleaning solutions
* High-pressure washes
* Abrasive materials
* Water or soap, which can damage the panels' electrical connections

Instead, use a gentle cleaning solution specifically designed for solar panels, or a mixture of 1 part white vinegar to 10 parts water. 
  RAG     : Acc

##### Context-converge score

In [8]:
fact_base = [fact_score(a, g) for a, g in zip(baseline_answers, ground_truths)]
fact_rag  = [fact_score(a, g) for a, g in zip(rag_answers, ground_truths)]
print(f"Key-number accuracy  baseline: {pd.Series(fact_base).mean():.2f}   RAG: {pd.Series(fact_rag).mean():.2f}")

Key-number accuracy  baseline: 0.34   RAG: 0.85
